# Interview Question Classifier — ruBERT-tiny2 → CoreML

**Pipeline:**
1. Скачать субтитры реальных русских IT-интервью с YouTube (`yt-dlp`)
2. Распарсить на реплики, авто-разметить эвристикой
3. Fine-tune `cointegrated/rubert-tiny2` (30 MB)
4. Конвертировать в CoreML `.mlpackage`

**Labels:** `technical_question` | `behavioral_question` | `directive` | `filler`

⚡ Runtime: GPU T4 (~15 мин обучение)

## 0. Установка зависимостей

In [ ]:
!pip install -q yt-dlp transformers datasets torch scikit-learn coremltools pandas numpy tqdm

## 1. Сбор субтитров с YouTube

In [ ]:
# ─── YouTube видео с русскими IT-интервью ──────────────────────────────────
# Вставь ID любых видео с интервью по программированию.
# Ищи по запросам: 'mock interview python', 'собеседование разработчик',
# 'системный дизайн интервью', 'алгоритмы интервью на русском'

VIDEO_IDS = [
    # Вставь YouTube video ID (часть после ?v=)
    # Пример: 'dQw4w9WgXcQ'
]

assert len(VIDEO_IDS) > 0, "Добавь хотя бы 5-10 video ID в список выше!"

In [ ]:
import subprocess, os, json
from pathlib import Path

SUBTITLES_DIR = Path('subtitles')
SUBTITLES_DIR.mkdir(exist_ok=True)

def download_subtitles(video_id: str) -> bool:
    """Скачивает авто-субтитры (ru) или ручные (ru/en) для видео."""
    url = f'https://www.youtube.com/watch?v={video_id}'
    out = SUBTITLES_DIR / video_id

    # Попытка 1: ручные русские субтитры
    # Попытка 2: автоматически сгенерированные русские
    for sub_flag in ['--write-subs --sub-lang ru', '--write-auto-subs --sub-lang ru']:
        cmd = (
            f'yt-dlp {sub_flag} --skip-download '
            f'--convert-subs vtt '
            f'-o "{out}/%(title)s.%(ext)s" "{url}"'
        )
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        vtt_files = list(out.glob('*.vtt'))
        if vtt_files:
            print(f'  ✅ {video_id}: {vtt_files[0].name}')
            return True

    print(f'  ❌ {video_id}: субтитры недоступны')
    return False

print('Скачиваем субтитры...')
ok = sum(download_subtitles(vid) for vid in VIDEO_IDS)
print(f'\n{ok}/{len(VIDEO_IDS)} видео с субтитрами')

## 2. Парсинг VTT → реплики

In [ ]:
import re
from dataclasses import dataclass
from typing import List

@dataclass
class Utterance:
    text: str
    start_sec: float
    source_video: str

def parse_vtt(vtt_path: Path) -> List[Utterance]:
    """Парсит .vtt файл в список реплик.
    Склеивает дублирующиеся строки (YouTube повторяет предыдущую строку).
    """
    content = vtt_path.read_text(encoding='utf-8')
    blocks = re.split(r'\n\n+', content)

    utterances = []
    seen_texts = set()

    for block in blocks:
        lines = block.strip().splitlines()
        # Ищем строку с таймкодом: 00:01:23.456 --> 00:01:25.789
        time_line = next((l for l in lines if '-->' in l), None)
        if not time_line:
            continue

        # Парсим start time
        start_str = time_line.split('-->')[0].strip().split()[0]
        parts = start_str.replace(',', '.').split(':')
        try:
            start_sec = sum(float(p) * 60**i for i, p in enumerate(reversed(parts)))
        except ValueError:
            continue

        # Текст — строки после таймкода, без HTML тегов
        text_lines = [l for l in lines if '-->' not in l and not l.isdigit()]
        text = re.sub(r'<[^>]+>', '', ' '.join(text_lines)).strip()
        text = re.sub(r'\s+', ' ', text)

        if not text or text in seen_texts or len(text) < 3:
            continue
        seen_texts.add(text)

        utterances.append(Utterance(
            text=text,
            start_sec=start_sec,
            source_video=vtt_path.parent.name
        ))

    return utterances

# ─── Склейка коротких реплик в смысловые блоки ────────────────────────────
def merge_short_utterances(utterances: List[Utterance],
                            min_words: int = 4,
                            max_gap_sec: float = 2.0) -> List[Utterance]:
    """Склеивает реплики <4 слов с соседними если пауза <2 сек."""
    if not utterances:
        return []

    merged = [utterances[0]]
    for u in utterances[1:]:
        prev = merged[-1]
        gap = u.start_sec - prev.start_sec
        if len(prev.text.split()) < min_words and gap < max_gap_sec:
            merged[-1] = Utterance(
                text=prev.text + ' ' + u.text,
                start_sec=prev.start_sec,
                source_video=prev.source_video
            )
        else:
            merged.append(u)
    return merged

# ─── Загрузка всех VTT ────────────────────────────────────────────────────
all_utterances: List[Utterance] = []
for vtt_file in SUBTITLES_DIR.rglob('*.vtt'):
    raw = parse_vtt(vtt_file)
    merged = merge_short_utterances(raw)
    all_utterances.extend(merged)
    print(f'{vtt_file.parent.name}: {len(raw)} → {len(merged)} реплик')

print(f'\nИтого реплик: {len(all_utterances)}')

## 3. Авто-разметка эвристикой

In [ ]:
import re

# ─── Паттерны для авто-разметки ───────────────────────────────────────────
TECHNICAL_KEYWORDS = re.compile(
    r'\b(алгоритм|сложность|паттерн|архитектур|базы? данных|sql|nosql|'
    r'deadlock|race condition|concurren|async|await|thread|process|'
    r'oop|solid|grasp|rest|api|http|tcp|udp|docker|kubernetes|git|ci|cd|'
    r'sort|search|hash|дерев|граф|стек|очередь|linked list|массив|'
    r'swift|kotlin|python|java|golang|rust|javascript|typescript|'
    r'memory leak|retain cycle|arc|gc|garbage|heap|stack|'
    r'microservic|monolith|scalab|балансировк|кэш|redis|kafka|'
    r'реализу|напиши|написать|реализовать)',
    re.IGNORECASE
)

BEHAVIORAL_KEYWORDS = re.compile(
    r'\b(расскажи|расскажите|опишите|опиши|ваш опыт|ваши|'
    r'почему вы|зачем вы|где вы|когда вы|как вы справля|'
    r'ситуаци|конфликт|команд|коллег|достижени|слабые сторон|'
    r'сильные сторон|ожидани|мотива|карьер|планируете)',
    re.IGNORECASE
)

DIRECTIVE_PATTERNS = re.compile(
    r'^(давайте|перейдём|переходим|следующий|хорошо,\s*(давайте|перейдём)|'
    r'откройте|открой|запустите|посмотрите|покажите)',
    re.IGNORECASE
)

FILLER_PATTERNS = re.compile(
    r'^(ага|угу|хмм?|окей|ок|да|нет|понятно|хорошо|ладно|'
    r'понял|поняла|ясно|интересно|отлично|супер|класс|'
    r'молодец|продолжайте|продолжай|да-да)[\.!,]?$',
    re.IGNORECASE
)

QUESTION_MARK = re.compile(r'[?？]')
QUESTION_WORDS = re.compile(
    r'\b(что|как|где|когда|зачем|почему|кто|какой|какая|какое|'
    r'сколько|чем|каким образом|можете ли|можешь ли|умеете)',
    re.IGNORECASE
)

def auto_label(text: str) -> str | None:
    """Возвращает метку или None если неоднозначно."""
    text = text.strip()

    # Филлер — очень короткие нейтральные реплики
    if FILLER_PATTERNS.match(text) or (len(text.split()) <= 2 and not QUESTION_MARK.search(text)):
        return 'filler'

    # Директива — начинается с команды
    if DIRECTIVE_PATTERNS.match(text) and not QUESTION_MARK.search(text):
        return 'directive'

    has_question = bool(QUESTION_MARK.search(text) or QUESTION_WORDS.search(text))

    # Поведенческий вопрос
    if BEHAVIORAL_KEYWORDS.search(text):
        return 'behavioral_question'

    # Технический вопрос
    if TECHNICAL_KEYWORDS.search(text) and has_question:
        return 'technical_question'

    # Общий вопрос (есть вопросительный знак или слово)
    if has_question:
        return 'technical_question'  # в IT-интервью большинство вопросов технические

    return None  # неоднозначно — попадёт в ручную проверку

# ─── Применяем авто-разметку ──────────────────────────────────────────────
labeled, ambiguous = [], []
for u in all_utterances:
    label = auto_label(u.text)
    if label:
        labeled.append({'text': u.text, 'label': label, 'source': u.source_video, 'auto': True})
    else:
        ambiguous.append({'text': u.text, 'source': u.source_video})

import pandas as pd
df_labeled   = pd.DataFrame(labeled)
df_ambiguous = pd.DataFrame(ambiguous)

print('Авто-разметка:')
print(df_labeled['label'].value_counts())
print(f'\nНеоднозначных (требуют ручной проверки): {len(df_ambiguous)}')

df_labeled.to_csv('labeled_auto.csv', index=False)
df_ambiguous.to_csv('to_review.csv', index=False)
print('\nСохранено: labeled_auto.csv, to_review.csv')

## 4. Ручная проверка + балансировка

Скачай `labeled_auto.csv` из панели Colab (Files → Download),  
проверь/исправь метки в Excel/Numbers, загрузи обратно как `labeled_final.csv`.  
Цель: **минимум 150 примеров на каждый класс**.

In [ ]:
# ─── Загрузка финального датасета ─────────────────────────────────────────
# Если не делал ручную проверку — используй авто-разметку как есть
import os

if os.path.exists('labeled_final.csv'):
    df = pd.read_csv('labeled_final.csv')
    print('Загружен labeled_final.csv (ручная проверка)')
else:
    df = df_labeled.copy()
    print('Используется авто-разметка (labeled_auto.csv)')

# Оставляем только нужные колонки
df = df[['text', 'label']].dropna()
df = df[df['text'].str.split().str.len() >= 2]  # убираем 1-словные

print('\nРаспределение классов:')
print(df['label'].value_counts())
print(f'\nИтого: {len(df)} примеров')

# Предупреждение если мало данных
min_count = df['label'].value_counts().min()
if min_count < 100:
    print(f'\n⚠️  Минимальный класс: {min_count} примеров. Рекомендуется >= 150.')
    print('    Добавь больше видео в VIDEO_IDS или исправь to_review.csv')

## 5. Обучение ruBERT-tiny2

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

MODEL_ID   = 'cointegrated/rubert-tiny2'
MAX_LEN    = 64
NUM_LABELS = df['label'].nunique()

labels     = sorted(df['label'].unique())
label2id   = {l: i for i, l in enumerate(labels)}
id2label   = {i: l for l, i in label2id.items()}

print('Классы:', label2id)

# ─── Train / Val split (стратифицированный) ───────────────────────────────
ds = Dataset.from_pandas(df[['text', 'label']])
ds = ds.train_test_split(test_size=0.2, stratify_by_column='label', seed=42)

# ─── Токенизация ──────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize(batch):
    enc = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN
    )
    enc['labels'] = [label2id[l] for l in batch['label']]
    return enc

tokenized = ds.map(tokenize, batched=True, remove_columns=['text', 'label'])
tokenized.set_format('torch')

# ─── Модель ───────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# ─── Метрики ──────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    preds  = np.argmax(eval_pred.predictions, axis=-1)
    labels = eval_pred.label_ids
    return {'accuracy': accuracy_score(labels, preds)}

# ─── Training args ────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir='./ckpt',
    num_train_epochs=15,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,          # T4 GPU
    logging_steps=20,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

trainer.train()
print(f'\n✅ Best eval accuracy: {trainer.state.best_metric:.4f}')

In [ ]:
# ─── Полный classification report ─────────────────────────────────────────
preds_output = trainer.predict(tokenized['test'])
preds        = np.argmax(preds_output.predictions, axis=-1)
true_labels  = preds_output.label_ids

print(classification_report(
    true_labels, preds,
    target_names=labels,
    digits=3
))

In [ ]:
# ─── Сохранение модели ────────────────────────────────────────────────────
SAVE_DIR = './question_classifier'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Сохраняем маппинг меток отдельно
with open(f'{SAVE_DIR}/id2label.json', 'w') as f:
    json.dump(id2label, f, ensure_ascii=False, indent=2)

print(f'Модель сохранена в {SAVE_DIR}/')

## 6. Конвертация в CoreML

In [ ]:
import torch
import coremltools as ct
import numpy as np

model.eval()

# ─── Wrapper: BERT → softmax probabilities ────────────────────────────────
class ClassifierWrapper(torch.nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        return torch.softmax(out.logits, dim=-1)

wrapper = ClassifierWrapper(model.cpu())
wrapper.eval()

# ─── TorchScript trace (фиксированный batch=1, len=64) ────────────────────
dummy_ids  = torch.zeros(1, MAX_LEN, dtype=torch.long)
dummy_mask = torch.ones(1,  MAX_LEN, dtype=torch.long)

with torch.no_grad():
    traced = torch.jit.trace(wrapper, (dummy_ids, dummy_mask))

print('TorchScript trace OK')

# ─── CoreML conversion ────────────────────────────────────────────────────
coreml = ct.convert(
    traced,
    inputs=[
        ct.TensorType(name='input_ids',      shape=(1, MAX_LEN), dtype=np.int32),
        ct.TensorType(name='attention_mask',  shape=(1, MAX_LEN), dtype=np.int32),
    ],
    outputs=[
        ct.TensorType(name='probabilities'),
    ],
    minimum_deployment_target=ct.target.macOS14,
    compute_units=ct.ComputeUnits.CPU_AND_NE,   # Neural Engine
)

# ─── Метаданные ───────────────────────────────────────────────────────────
coreml.short_description = 'Interview question classifier (rubert-tiny2)'
coreml.input_description['input_ids']      = f'WordPiece token IDs, padded to {MAX_LEN}'
coreml.input_description['attention_mask'] = f'Attention mask, padded to {MAX_LEN}'
coreml.output_description['probabilities'] = f'Class probabilities [{", ".join(labels)}]'

# Вшиваем labels и max_len в модель — читаются в Swift без отдельных файлов
coreml.user_defined_metadata['labels']  = json.dumps(id2label)
coreml.user_defined_metadata['max_len'] = str(MAX_LEN)
coreml.user_defined_metadata['labels_order'] = json.dumps(labels)  # порядок для вектора

COREML_PATH = 'QuestionClassifier.mlpackage'
coreml.save(COREML_PATH)

import os
size_mb = sum(f.stat().st_size for f in Path(COREML_PATH).rglob('*') if f.is_file()) / 1e6
print(f'\n✅ Сохранено: {COREML_PATH} ({size_mb:.1f} MB)')

## 7. Быстрая проверка модели

In [ ]:
import coremltools as ct
import numpy as np

# Загружаем обратно и проверяем
loaded = ct.models.MLModel(COREML_PATH)

def predict_coreml(text: str) -> dict:
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='np'
    )
    out = loaded.predict({
        'input_ids':     enc['input_ids'].astype(np.int32),
        'attention_mask': enc['attention_mask'].astype(np.int32),
    })
    probs = out['probabilities'].flatten()
    best  = int(np.argmax(probs))
    return {'label': id2label[best], 'confidence': float(probs[best])}

# ─── Smoke test ───────────────────────────────────────────────────────────
tests = [
    ('Что такое deadlock и как его избежать?',         'technical_question'),
    ('Объясните принцип единственной ответственности', 'technical_question'),
    ('Расскажите о вашем опыте работы в команде',      'behavioral_question'),
    ('Почему вы хотите сменить работу?',               'behavioral_question'),
    ('Давайте перейдём к следующей теме',              'directive'),
    ('Угу, понятно',                                   'filler'),
    ('Да',                                             'filler'),
]

print(f'{"Текст":<50} {"Ожидание":<22} {"Результат":<22} {"Conf"}')
print('-' * 105)
correct = 0
for text, expected in tests:
    r = predict_coreml(text)
    ok = '✅' if r['label'] == expected else '❌'
    if r['label'] == expected:
        correct += 1
    print(f"{ok} {text[:48]:<50} {expected:<22} {r['label']:<22} {r['confidence']:.2f}")

print(f'\nSmoke test: {correct}/{len(tests)}')

## 8. Скачать артефакты

Скачай следующие файлы из Colab (`Files` → правая кнопка → Download):
- `QuestionClassifier.mlpackage` → в Xcode проект (`AIssistant/Resources/Models/`)
- `question_classifier/vocab.txt` → `AIssistant/Resources/` (для Swift токенизатора)
- `question_classifier/tokenizer_config.json` → `AIssistant/Resources/`

In [ ]:
# ─── Архивируем всё для скачивания ───────────────────────────────────────
!zip -r question_classifier_artifacts.zip \
    QuestionClassifier.mlpackage \
    question_classifier/vocab.txt \
    question_classifier/tokenizer_config.json \
    question_classifier/id2label.json \
    labeled_auto.csv

print('\n✅ Архив готов: question_classifier_artifacts.zip')
print('Скачай его из панели Files слева.')